In [ ]:
from google import genai

# Replace with your actual API key
client = genai.Client(api_key="YOUR_GEMINI_API_KEY")

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents="I am testing the API key"
)

print(response.text)


#    - ONE sentence describing the interaction dynamic.
#    - Mention both the AI and User roles. Either may appear first depending on who initiated.
#    - Do NOT use newlines. Do NOT write multiple sentences.

: 

In [19]:
from openai import OpenAI

# Replace 'your-api-key-here' with your actual key
client = OpenAI(api_key="YOUR_OPENAI_API_KEY")


try:
    response = client.chat.completions.create(
        model="gpt-4o-mini", # Or "gpt-4o" if available
        messages=[{"role": "user", "content": "This is a test message."}]
    )
    print("Success! Response:", response.choices[0].message.content)
except Exception as e:
    print(f"Error: {e}")


Success! Response: Test received! How can I assist you today?


# The important code

In [3]:
# =============================================================================
# CONVERSATION EXTRACTOR V3 - MULTI-PROVIDER PARALLEL PROCESSING
# =============================================================================
# This script extracts structured strategic insights from dawah conversations
# using parallel processing. Supports both OpenAI and Google Gemini models.
# =============================================================================


# =============================================================================
# CELL 1: Installation
# =============================================================================
# Run this cell first (uncomment if needed)

# !pip install openai pandas json-repair tqdm python-dotenv google-generativeai


# =============================================================================
# CELL 2: Imports
# =============================================================================

import json
import os
import pandas as pd
from json_repair import repair_json
from tqdm import tqdm
import warnings
import ast
from datetime import datetime
import statistics
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from google.genai import types as genai_types
import concurrent.futures
import threading
import time

warnings.filterwarnings("ignore")

# Load environment variables from .env file (if exists)
load_dotenv()

print("All imports successful!")



All imports successful!


In [4]:

# =============================================================================
# CELL 3: Configuration
# =============================================================================

# ============================================
# PROVIDER & MODEL SETTINGS (from .env)
# ============================================
PROVIDER = os.environ.get("LLM_PROVIDER", "openai").lower()   # "openai" or "gemini"
MODEL_ID = os.environ.get("LLM_MODEL", "gpt-4o-mini")

# ============================================
# MODEL PRICING (per 1M tokens, as of April 2026)
# ============================================
MODEL_PRICING = {
    # ===================== OpenAI =====================
    "gpt-4o-mini":       {"input": 0.15,  "output": 0.60},
    "gpt-5":             {"input": 2.00,  "output": 15.00},
    "gpt-5-mini":        {"input": 0.25,  "output": 0.60},
    "gpt-5-nano":        {"input": 0.10,  "output": 0.40},
    "o3":                {"input": 2.00,  "output": 8.00},
    "o3-mini":           {"input": 1.10,  "output": 4.40},
    # ===================== Gemini 3.x (Latest) =====================
    "gemini-3.1-pro-preview":            {"input": 2.00, "output": 12.00},
    "gemini-3-flash-preview":            {"input": 0.50, "output": 3.00},
    "gemini-3.1-flash-lite-preview":     {"input": 0.25, "output": 1.50},
    # ===================== Gemini 2.5 (Stable) =====================
    "gemini-2.5-pro":                    {"input": 1.25, "output": 10.00},
    "gemini-2.5-flash":                  {"input": 0.30, "output": 2.50},
    "gemini-2.5-flash-lite":             {"input": 0.10, "output": 0.40},
    "gemini-2.5-flash-lite-preview-09-2025": {"input": 0.10, "output": 0.40},
    # ===================== Gemini 2.0 (Deprecated) =====================
    "gemini-2.0-flash":                  {"input": 0.10, "output": 0.40},
    "gemini-2.0-flash-lite":             {"input": 0.075, "output": 0.30},
}

# ============================================
# PROCESSING MODE
# ============================================
PROCESSING_MODE = os.environ.get("PROCESSING_MODE", "parallel").lower()  # "parallel" or "batch"
# - "parallel": Real-time processing with ThreadPoolExecutor (current behavior)
# - "batch":    Gemini Batch API — 50% cost, async (submit & poll, up to 24h)

# ============================================
# PARALLEL PROCESSING SETTINGS
# ============================================
NUM_WORKERS = 16  # Number of parallel workers (recommended: 5-20)
                  # Adjust based on your API rate limits:
                  # - Tier 1: Use 5-10 workers
                  # - Tier 2+: Use 10-20 workers

# ============================================
# FILE PATHS - UPDATE THESE
# ============================================
INPUT_FILE = r"S:\Midade work\Islam chat conversation analysis\organized_conversations_with_language.csv"
OUTPUT_FILE = f"full_conversation_points_extraction.csv"

# ============================================
# CHECKPOINTING SETTINGS
# ============================================
CHECKPOINT_FILE = f"checkpoint_{MODEL_ID.replace('.', '_').replace('-', '_')}.json"
CHECKPOINT_EVERY = 10  # Save progress every N completed rows

# ============================================
# BATCH MODE SETTINGS
# ============================================
BATCH_JOB_FILE = f"batch_job_{MODEL_ID.replace('.', '_').replace('-', '_')}.json"  # Stores job name for resume
BATCH_JSONL_FILE = "batch_requests.jsonl"  # Temp JSONL file for batch input
BATCH_POLL_INTERVAL = 30  # Seconds between status polls

# ============================================
# GENERATION SETTINGS
# ============================================
MAX_TOKENS = 1024
TEMPERATURE = 0.1  # Low temperature for consistent classification

# ============================================
# RETRY SETTINGS
# ============================================
MAX_RETRIES = 5
RETRY_BASE_DELAY = 1  # Base delay in seconds for exponential backoff

# ============================================
# VALIDATION
# ============================================
if MODEL_ID not in MODEL_PRICING:
    print(f"⚠️  WARNING: Model '{MODEL_ID}' not found in MODEL_PRICING dictionary.")
    print(f"   Cost estimation will show $0. Add pricing to MODEL_PRICING if needed.")

# Validate batch mode requires Gemini
if PROCESSING_MODE == "batch" and PROVIDER != "gemini":
    raise ValueError(
        f"Batch mode only supports Gemini provider, but LLM_PROVIDER='{PROVIDER}'.\n"
        f"Either set LLM_PROVIDER=gemini or PROCESSING_MODE=parallel."
    )

print(f"Provider: {PROVIDER.upper()}")
print(f"Selected Model: {MODEL_ID}")
print(f"Processing Mode: {PROCESSING_MODE.upper()}")
if PROCESSING_MODE == "parallel":
    print(f"Parallel Workers: {NUM_WORKERS}")
print(f"Input File: {INPUT_FILE}")
print(f"Output File: {OUTPUT_FILE}")
print(f"Checkpoint File: {CHECKPOINT_FILE}")



Provider: GEMINI
Selected Model: gemini-3.1-flash-lite-preview
Processing Mode: BATCH
Input File: S:\Midade work\Islam chat conversation analysis\organized_conversations_with_language.csv
Output File: full_conversation_points_extraction.csv
Checkpoint File: checkpoint_gemini_3_1_flash_lite_preview.json


In [5]:

# =============================================================================
# CELL 4: Initialize LLM Client (Provider-Aware)
# =============================================================================

if PROVIDER == "openai":
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY not found. Set it in .env or environment variables.")
    client = OpenAI(api_key=api_key)
    print(f"✅ OpenAI client initialized! Model: {MODEL_ID}")

elif PROVIDER == "gemini":
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("GOOGLE_API_KEY not found. Set it in .env or environment variables.")
    client = genai.Client(api_key=api_key)
    print(f"✅ Gemini client initialized (google-genai SDK)! Model: {MODEL_ID}")

else:
    raise ValueError(f"Unknown provider '{PROVIDER}'. Use 'openai' or 'gemini'.")


# =============================================================================
# CELL 5: Extraction Prompt Template (V3)
# =============================================================================

EXTRACT_SYSTEM_PROMPT = """You are an expert Data Analyst for an Islamic missionary (Dawah) organization.
Your goal is to audit conversations between a Dawah Bot ("AI") and human users ("User").
You must be critical and objective. Do NOT conflate "Bot Preaching" with "User Interest."
A user who only says "ok" or "yes" to a long bot script is NOT engaged — they are passive.
If the user is already Muslim, adjust your analysis accordingly — they are not a conversion prospect.
Output strict JSON only. No markdown, no commentary. Translate ALL non-English extracted values to English."""

EXTRACT_USER_PROMPT_TEMPLATE = """### TASK:
Analyze this conversation between a Dawah Bot ("AI") and a human ("User").
The AI uses pre-set scripts and may dominate the conversation. The User may be a non-Muslim prospect, an existing Muslim, a troll, or anything else.
Your job is to extract strategic insights, filtering out low-effort interactions and bot noise.

### CONVERSATION DATA:
{conversation}

### EXTRACTION RULES:

1. **conversation_summary**:
   - Write a NARRATIVE summary (3-8 sentences) that tells the full story of the conversation.
   - Cover: who the user appears to be, what topics were raised, how the conversation
     developed, any shifts in tone or topic, and how it ended.
   - Include any notable suspicions (e.g., user might be testing the bot, user seems
     to already know Islam well, user might be trolling, user switched languages).
   - LENGTH RULE: Scale with conversation length.
     * Short conversations (1-5 messages): 2-3 sentences.
     * Medium conversations (6-15 messages): 4-5 sentences.
     * Long conversations (16+ messages): 5-8 sentences.
   - Do NOT use newlines inside the summary string. Write it as one continuous paragraph.
   - Do NOT be vague. WRONG: "User asked questions and bot answered."
     RIGHT: "A user who appears to be Christian asked about the Trinity and why Islam
     rejects it. The bot explained tawhid with Quranic references. The user pushed back
     citing John 1:1, and the conversation became a theological debate. The user eventually
     stopped responding after the bot's third response."

2. **user_demographics**:
   - "suspected_religion": The user's religion. First check if the user explicitly stated it
     (the AI often asks directly). If not stated, infer from the conversation content, vocabulary,
     and worldview. Choose from: "Christianity", "Atheism", "Hinduism", "Buddhism", "Judaism",
     "Islam", "Agnosticism", "Irreligion", "Other", "Unknown".
   - "is_existing_muslim": true if the user explicitly identifies as Muslim OR asks Muslim-specific
     questions (fiqh, worship, Quran recitation). false otherwise.
   - "language": The primary language the USER writes in (not the bot's language).
     If User writes in multiple languages, pick the dominant one.

3. **conversation_type**: Choose ONE:
   * "Dawah" — AI is introducing Islam to a non-Muslim or answering a non-Muslim's questions about Islam.
   * "Theological Debate" — User challenges, argues against, or criticizes Islamic claims.
   * "Islamic Guidance" — User asks about Islamic rules, worship, Quran, hadith, sectarian questions, or daily Muslim life.
   * "Emotional Support" — User shares personal struggles, hardships, or seeks comfort and encouragement.
   * "Content Assistance" — User asks the bot to write, rephrase, summarize, or generate content.
   * "Dawah Training" — Muslim user asks for help responding to non-Muslim arguments or preparing dawah material.
   * "Off-Topic" — Conversation is unrelated to Islam or dawah (legal, tech, general knowledge, etc.).
   * "Minimal/No Engagement" — Greeting only, gibberish, single message with no follow-up, or bot-only response.

4. **engagement_quality** (CRITICAL — be strict):
   - "score": Integer 1-5.
     * 1 = No real interaction. Gibberish, emojis, or user never replied after the first message.
     * 2 = Minimal. User only says "Yes", "Ok", "Amen", or gives one-word answers to bot scripts.
     * 3 = Reactive. User answers bot's questions with short but meaningful sentences.
     * 4 = Active. User asks their own questions, shares opinions, or raises objections.
     * 5 = Deep. Extended back-and-forth with substantial exchanges from both sides.
   - "flow": Choose from: "Bot-Dominated", "Balanced", "User-Dominated".

5. **theological_profile**:
   - "topics_discussed": List of specific subjects discussed in the conversation.
     * INCLUSION: A topic counts if the User introduced it OR meaningfully responded to it.
     * SPECIFICITY: Use precise sub-topics (e.g., "Trinity", "Ramadan Fasting", "Hadith Authenticity"),
       not vague labels (e.g., "Islam", "Religion").
     * EXCLUSION: Do NOT list topics from bot scripts that the User completely ignored.
     * If no real topics were discussed, return [].
   - "key_blocker": The primary intellectual, emotional, or cultural barrier preventing the user
     from accepting or exploring Islam further.
     Examples: "Trinity", "Atheism", "Cultural Identity", "Logical Skepticism",
     "Negative Media Perception", "Family Pressure", "Distrust of Religion",
     "Moral Objection", "Lack of Interest", "Sectarian Confusion".
     If user is already Muslim or shows zero resistance, return "N/A".
   - "user_objections": List of counter-arguments, criticisms, or pushback the user gave.
     These are different from questions — objections challenge a claim rather than ask about it.
     If none, return [].

6. **emotional_trajectory**:
   - "start_mood": The user's emotional state in their first messages
     (e.g., Curious, Neutral, Skeptical, Hostile, Anxious, Enthusiastic,
     Confused, Distressed, Playful, Reverent, Confident, Respectful).
     Use "Neutral" when the user simply states facts, greets, or asks a basic question
     without emotional tone. Use "Curious" ONLY when the user shows active, clear interest.
   - "end_mood": The user's emotional state in their last messages
     (e.g., Inspired, Informed, Engaged, Satisfied, Frustrated, Neutral,
     Ghosted, Skeptical, Grateful, Receptive, Overwhelmed, Indifferent, Comforted).
     Use "Ghosted" when the user stopped responding and never came back.

7. **intent_and_funnel**:
   - "user_intent": Choose ONE:
     * "Genuine Seeker" — Non-Muslim asking real questions about Islam with sincere interest.
     * "Passive Listener" — User gives minimal responses ("ok", "yes", short answers) regardless
       of their religion. A Muslim who only says "Ameen" to bot's dua is STILL a Passive Listener.
       RULE: If the user has engagement_score <= 2 AND did not ask any questions, use this.
     * "Conversion Interest" — Explicitly expresses desire to convert or takes shahada.
     * "Challenger" — Argues against, debates, or criticizes Islamic claims.
     * "Troll/Spam" — Sends gibberish, sarcasm, mockery, or spam.
     * "Greeting Only" — Only Hi/Hello/Salam, no substantive conversation.
     * "Muslim Learner" — Already Muslim AND actively engaged (asks questions, discusses topics).
       RULE: Only use this if is_existing_muslim=true AND engagement_score >= 3.
     * "Off-Topic User" — Asking about topics unrelated to Islam.
   - "conversion_funnel": Applies ONLY to non-Muslim users in dawah-related conversations.
     * "Top" — Awareness stage, first contact or basic curiosity.
     * "Middle" — Consideration, asking deeper questions or engaging with arguments.
     * "Bottom" — Decision stage, expressing readiness or strong inclination.
     * "Converted" — User took shahada or explicitly declared conversion.
     * "Dropped" — User left, rejected, or lost interest.
     * "N/A" — MUST use if ANY of: is_existing_muslim=true, conversation_type is not "Dawah",
       user_intent is "Greeting Only" / "Troll/Spam" / "Off-Topic User" / "Muslim Learner".

8. **bot_quality_audit**:
   - "script_dumping": Did the bot paste long pre-written scripts ignoring user input? (true/false).
     CRITERIA — ALL THREE must be true for script_dumping=true:
     (a) Bot sent at least 3 messages that are each longer than the user's longest message.
     (b) Bot messages feel generic / copy-pasted (not tailored to what the user said).
     (c) Bot continues its script even when the user changes topic or asks a specific question.
     If only (a) is true but the bot is genuinely responding to the user, return false.
   - "response_quality": Integer 1-5 rating of how well the bot handled this conversation.
     * 1 = Completely irrelevant, broken, or ignored the user's actual question.
     * 2 = Addressed the wrong topic or missed the user's point entirely.
     * 3 = Adequate but generic — could apply to any conversation.
     * 4 = Good, relevant, and showed awareness of the user's specific situation.
     * 5 = Excellent, personalized, insightful, and contextually appropriate.
   - "critique": ONE specific sentence about the bot's most notable failure or weakness.
     If the bot performed well, return "No major issues."
     WRONG: "Bot provided relevant information." (too vague, not useful)
     RIGHT: "Bot sent a 500-word script when user only asked a yes/no question."
     RIGHT: "Bot asked for nationality after user already said they are from Morocco."
     RIGHT: "Bot switched to Arabic when user was writing in French."

### JSON OUTPUT (Strict JSON only, no markdown wrapping):
{{
  "conversation_summary": "string",
  "user_demographics": {{
    "suspected_religion": "string",
    "is_existing_muslim": false,
    "language": "string"
  }},
  "conversation_type": "string",
  "engagement_quality": {{
    "score": 0,
    "flow": "string"
  }},
  "theological_profile": {{
    "topics_discussed": ["string"],
    "key_blocker": "string",
    "user_objections": ["string"]
  }},
  "emotional_trajectory": {{
    "start_mood": "string",
    "end_mood": "string"
  }},
  "intent_and_funnel": {{
    "user_intent": "string",
    "conversion_funnel": "string"
  }},
  "bot_quality_audit": {{
    "script_dumping": false,
    "response_quality": 0,
    "critique": "string"
  }}
}}"""

print("✅ Extraction prompt loaded (V3)!")



Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Gemini client initialized (google-genai SDK)! Model: gemini-3.1-flash-lite-preview
✅ Extraction prompt loaded (V3)!


In [6]:

# =============================================================================
# CELL 6: LLM API Call Function (Provider-Aware) with Token Tracking
# =============================================================================

def call_llm_api(system_prompt, user_prompt, max_retries=MAX_RETRIES):
    """
    Call the configured LLM API and return response with token counts.
    Supports both OpenAI and Gemini providers.
    Includes exponential backoff for rate limiting.
    Returns: (response_text, input_tokens, output_tokens)
    """
    for attempt in range(max_retries):
        try:
            if PROVIDER == "openai":
                response = client.chat.completions.create(
                    model=MODEL_ID,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    response_format={"type": "json_object"}
                )
                
                response_text = response.choices[0].message.content.strip()
                input_tokens = response.usage.prompt_tokens
                output_tokens = response.usage.completion_tokens
                
            elif PROVIDER == "gemini":
                response = client.models.generate_content(
                    model=MODEL_ID,
                    contents=user_prompt,
                    config=genai_types.GenerateContentConfig(
                        system_instruction=system_prompt,
                        temperature=TEMPERATURE,
                        max_output_tokens=MAX_TOKENS,
                        response_mime_type="application/json"
                    )
                )
                
                response_text = response.text.strip()
                # Extract token counts from Gemini usage metadata
                usage = response.usage_metadata
                input_tokens = usage.prompt_token_count
                output_tokens = usage.candidates_token_count
            
            return response_text, input_tokens, output_tokens
            
        except Exception as e:
            error_str = str(e).lower()
            
            # Check if it's a rate limit error
            if 'rate' in error_str or '429' in error_str or 'quota' in error_str or 'resource_exhausted' in error_str:
                wait_time = RETRY_BASE_DELAY * (2 ** attempt) + (attempt * 0.5)
                print(f"   ⚠️ Rate limit hit, waiting {wait_time:.1f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"   ⚠️ API error (attempt {attempt + 1}/{max_retries}): {e}")
                if attempt < max_retries - 1:
                    time.sleep(RETRY_BASE_DELAY * (2 ** attempt))
                else:
                    raise e


print("✅ LLM API function loaded!")



✅ LLM API function loaded!


In [7]:

# =============================================================================
# CELL 7: Core Processing Functions
# =============================================================================

def parse_json_response(response):
    """Parse and validate JSON response from extraction."""
    
    response = response.strip()
    
    # Remove markdown code blocks if present
    if response.startswith("```json"):
        response = response[7:]
    if response.startswith("```"):
        response = response[3:]
    if response.endswith("```"):
        response = response[:-3]
    
    response = response.strip()
    
    try:
        repaired = repair_json(response)
        result = json.loads(repaired)
        
        # Validate top-level required fields
        required_top_fields = [
            "conversation_summary", "user_demographics", "conversation_type",
            "engagement_quality", "theological_profile", "emotional_trajectory",
            "intent_and_funnel", "bot_quality_audit"
        ]
        
        for field in required_top_fields:
            if field not in result:
                result[field] = {}
        
        # Ensure conversation_type is a string
        if not isinstance(result.get("conversation_type"), str):
            result["conversation_type"] = "Minimal/No Engagement"
        
        # Ensure nested structures exist with defaults
        if not isinstance(result.get("user_demographics"), dict):
            result["user_demographics"] = {}
        result["user_demographics"].setdefault("suspected_religion", "Unknown")
        result["user_demographics"].setdefault("is_existing_muslim", False)
        result["user_demographics"].setdefault("language", "Unknown")
        
        if not isinstance(result.get("engagement_quality"), dict):
            result["engagement_quality"] = {}
        result["engagement_quality"].setdefault("score", 0)
        result["engagement_quality"].setdefault("flow", "Unknown")
        
        if not isinstance(result.get("theological_profile"), dict):
            result["theological_profile"] = {}
        result["theological_profile"].setdefault("topics_discussed", [])
        result["theological_profile"].setdefault("key_blocker", "N/A")
        result["theological_profile"].setdefault("user_objections", [])
        
        if not isinstance(result.get("emotional_trajectory"), dict):
            result["emotional_trajectory"] = {}
        result["emotional_trajectory"].setdefault("start_mood", "Unknown")
        result["emotional_trajectory"].setdefault("end_mood", "Unknown")
        
        if not isinstance(result.get("intent_and_funnel"), dict):
            result["intent_and_funnel"] = {}
        result["intent_and_funnel"].setdefault("user_intent", "Unknown")
        result["intent_and_funnel"].setdefault("conversion_funnel", "Unknown")
        
        if not isinstance(result.get("bot_quality_audit"), dict):
            result["bot_quality_audit"] = {}
        result["bot_quality_audit"].setdefault("script_dumping", False)
        result["bot_quality_audit"].setdefault("response_quality", 0)
        result["bot_quality_audit"].setdefault("critique", "")
        
        # Fix empty lists stored as strings
        if result["theological_profile"]["topics_discussed"] == "None":
            result["theological_profile"]["topics_discussed"] = []
        if result["theological_profile"]["user_objections"] == "None":
            result["theological_profile"]["user_objections"] = []
        
        result["extraction_status"] = "success"
        
    except Exception as e:
        result = {
            "conversation_summary": "EXTRACTION_ERROR",
            "user_demographics": {"suspected_religion": "ERROR", "is_existing_muslim": False, "language": "ERROR"},
            "conversation_type": "ERROR",
            "engagement_quality": {"score": 0, "flow": "ERROR"},
            "theological_profile": {"topics_discussed": [], "key_blocker": "ERROR", "user_objections": []},
            "emotional_trajectory": {"start_mood": "ERROR", "end_mood": "ERROR"},
            "intent_and_funnel": {"user_intent": "ERROR", "conversion_funnel": "ERROR"},
            "bot_quality_audit": {"script_dumping": False, "response_quality": 0, "critique": "ERROR"},
            "extraction_status": "failed",
            "error_message": str(e)
        }
    
    return result


def flatten_extraction_result(extracted):
    """
    Flatten nested JSON extraction result into a flat dictionary for DataFrame.
    """
    flat = {
        # Top level
        "conversation_summary": extracted.get("conversation_summary", ""),
        
        # user_demographics
        "suspected_religion": extracted.get("user_demographics", {}).get("suspected_religion", "Unknown"),
        "is_existing_muslim": extracted.get("user_demographics", {}).get("is_existing_muslim", False),
        "user_language": extracted.get("user_demographics", {}).get("language", "Unknown"),
        
        # conversation_type
        "conversation_type": extracted.get("conversation_type", ""),
        
        # engagement_quality
        "engagement_score": extracted.get("engagement_quality", {}).get("score", 0),
        "engagement_flow": extracted.get("engagement_quality", {}).get("flow", "Unknown"),
        
        # theological_profile
        "topics_discussed": json.dumps(extracted.get("theological_profile", {}).get("topics_discussed", [])),
        "key_blocker": extracted.get("theological_profile", {}).get("key_blocker", "N/A"),
        "user_objections": json.dumps(extracted.get("theological_profile", {}).get("user_objections", [])),
        
        # emotional_trajectory
        "start_mood": extracted.get("emotional_trajectory", {}).get("start_mood", "Unknown"),
        "end_mood": extracted.get("emotional_trajectory", {}).get("end_mood", "Unknown"),
        
        # intent_and_funnel
        "user_intent": extracted.get("intent_and_funnel", {}).get("user_intent", "Unknown"),
        "conversion_funnel": extracted.get("intent_and_funnel", {}).get("conversion_funnel", "Unknown"),
        
        # bot_quality_audit
        "script_dumping": extracted.get("bot_quality_audit", {}).get("script_dumping", False),
        "response_quality": extracted.get("bot_quality_audit", {}).get("response_quality", 0),
        "bot_critique": extracted.get("bot_quality_audit", {}).get("critique", ""),
        
        # Status
        "extraction_status": extracted.get("extraction_status", "unknown")
    }
    
    return flat


def extract_from_conversation(conversation_text):
    """
    Extract structured points directly from a conversation.
    Returns: (parsed_result, input_tokens, output_tokens)
    """
    user_content = EXTRACT_USER_PROMPT_TEMPLATE.format(
        conversation=conversation_text
    )
    
    response_text, input_tokens, output_tokens = call_llm_api(
        EXTRACT_SYSTEM_PROMPT, user_content
    )
    
    parsed_result = parse_json_response(response_text)
    
    return parsed_result, input_tokens, output_tokens


def process_single_row(row_data):
    """
    Process a single conversation: extract strategic insights.
    Designed for parallel execution.
    
    Args:
        row_data: tuple of (index, row_dict)
    
    Returns: 
        tuple of (index, result_dict)
    """
    idx, row = row_data
    
    conversation_id = row.get('general_chat_id', 'unknown')
    conversation = str(row.get('full_conversation', ''))
    
    # Initialize result with all expected columns
    result = {
        'conversation_id': conversation_id,
        # Extracted fields (flattened)
        'conversation_summary': '',
        'suspected_religion': '',
        'is_existing_muslim': False,
        'user_language': '',
        'conversation_type': '',
        'engagement_score': 0,
        'engagement_flow': '',
        'topics_discussed': '[]',
        'key_blocker': '',
        'user_objections': '[]',
        'start_mood': '',
        'end_mood': '',
        'user_intent': '',
        'conversion_funnel': '',
        'script_dumping': False,
        'response_quality': 0,
        'bot_critique': '',
        # Processing metadata
        'processing_status': 'pending',
        'input_tokens': 0,
        'output_tokens': 0,
        'total_tokens': 0
    }
    
    try:
        # Extract insights directly from conversation
        extracted, input_tokens, output_tokens = extract_from_conversation(conversation)
        
        # Flatten the nested result
        flat_result = flatten_extraction_result(extracted)
        
        # Update result with flattened data
        result.update(flat_result)
        result['processing_status'] = flat_result.get('extraction_status', 'success')
        
        # Token counts
        result['input_tokens'] = input_tokens
        result['output_tokens'] = output_tokens
        result['total_tokens'] = input_tokens + output_tokens
            
    except Exception as e:
        result['processing_status'] = f'error: {str(e)}'
    
    return (idx, result)


print("✅ Core functions loaded!")



✅ Core functions loaded!


In [8]:

# =============================================================================
# CELL 8: Thread-Safe Checkpointing Functions
# =============================================================================

# Thread-safe lock for checkpointing
checkpoint_lock = threading.Lock()


def load_checkpoint():
    """Load checkpoint if exists, return processed results."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
                checkpoint = json.load(f)
            print(f"✅ Checkpoint found! {len(checkpoint['completed_indices'])} rows already processed")
            return checkpoint
        except Exception as e:
            print(f"⚠️ Checkpoint file corrupted, starting fresh: {e}")
            return None
    return None


def save_checkpoint(completed_indices, results_dict):
    """
    Thread-safe checkpoint saving.
    
    Args:
        completed_indices: set of completed row indices
        results_dict: dict mapping index to result
    """
    with checkpoint_lock:
        checkpoint = {
            'completed_indices': list(completed_indices),
            'results': {str(k): v for k, v in results_dict.items()},
            'model_id': MODEL_ID,
            'timestamp': datetime.now().isoformat()
        }
        
        # Save to temp file first, then rename (atomic operation)
        temp_file = CHECKPOINT_FILE + ".tmp"
        with open(temp_file, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False)
        
        os.replace(temp_file, CHECKPOINT_FILE)


def delete_checkpoint():
    """Delete checkpoint file after successful completion."""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("✅ Checkpoint file cleaned up")


print("✅ Thread-safe checkpointing functions loaded!")



✅ Thread-safe checkpointing functions loaded!


In [9]:

# =============================================================================
# CELL 9: Main Processing (Mode-Aware)
# =============================================================================

# Load data (shared by both modes)
print("="*60)
print(f"STARTING EXTRACTION WITH {MODEL_ID.upper()}")
print(f"Provider: {PROVIDER.upper()}")
print(f"Mode: {PROCESSING_MODE.upper()}")
print("="*60)

print(f"\nLoading data from: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE)
total_rows = len(df)
print(f"Total conversations: {total_rows}")

start_time = time.time()


# =============================================================================
# CELL 9A: PARALLEL MODE (existing behavior)
# =============================================================================

if PROCESSING_MODE == "parallel":
    print(f"\n⚡ PARALLEL MODE — {NUM_WORKERS} workers")
    print("-"*60)

    # Check for checkpoint
    checkpoint = load_checkpoint()

    if checkpoint:
        # Verify checkpoint is for the same model
        if checkpoint.get('model_id') != MODEL_ID:
            print(f"⚠️ Checkpoint is for different model ({checkpoint.get('model_id')})")
            print(f"   Current model: {MODEL_ID}")
            user_input = input("   Start fresh? (y/n): ")
            if user_input.lower() == 'y':
                completed_indices = set()
                results_dict = {}
            else:
                completed_indices = set(checkpoint['completed_indices'])
                results_dict = {int(k): v for k, v in checkpoint['results'].items()}
        else:
            completed_indices = set(checkpoint['completed_indices'])
            results_dict = {int(k): v for k, v in checkpoint['results'].items()}
            print(f"   Resuming... {len(completed_indices)} already done, {total_rows - len(completed_indices)} remaining")
    else:
        completed_indices = set()
        results_dict = {}

    # Prepare rows to process (skip already completed)
    rows_to_process = []
    for idx in range(total_rows):
        if idx not in completed_indices:
            row = df.iloc[idx].to_dict()
            rows_to_process.append((idx, row))

    print(f"\nRows to process: {len(rows_to_process)}")
    print("-"*60)

    # Tracking variables
    processed_count = len(completed_indices)
    last_checkpoint_count = processed_count

    # Progress bar
    pbar = tqdm(total=total_rows, initial=processed_count, desc="Extracting")

    # Thread-safe counter
    counter_lock = threading.Lock()


    def update_progress(idx, result):
        """Thread-safe progress update."""
        global processed_count, last_checkpoint_count
        
        with counter_lock:
            results_dict[idx] = result
            completed_indices.add(idx)
            processed_count += 1
            pbar.update(1)
            
            # Show brief status
            conv_id = result['conversation_id']
            status = result['processing_status']
            intent = result['user_intent']
            pbar.set_postfix({
                'ID': str(conv_id)[:10],
                'Intent': intent[:15] if intent else 'N/A',
                'Status': 'OK' if status == 'success' else 'ERR'
            })
            
            # Save checkpoint periodically
            if processed_count - last_checkpoint_count >= CHECKPOINT_EVERY:
                save_checkpoint(completed_indices, results_dict)
                last_checkpoint_count = processed_count


    # Process in parallel
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
            # Submit all tasks
            future_to_idx = {
                executor.submit(process_single_row, row_data): row_data[0] 
                for row_data in rows_to_process
            }
            
            # Process completed tasks
            for future in concurrent.futures.as_completed(future_to_idx):
                try:
                    idx, result = future.result()
                    update_progress(idx, result)
                except Exception as e:
                    idx = future_to_idx[future]
                    error_result = {
                        'conversation_id': df.iloc[idx].to_dict().get('general_chat_id', 'unknown'),
                        'conversation_summary': '',
                        'suspected_religion': '',
                        'is_existing_muslim': False,
                        'user_language': '',
                        'conversation_type': '',
                        'engagement_score': 0,
                        'engagement_flow': '',
                        'topics_discussed': '[]',
                        'key_blocker': '',
                        'user_objections': '[]',
                        'start_mood': '',
                        'end_mood': '',
                        'user_intent': '',
                        'conversion_funnel': '',
                        'script_dumping': False,
                        'response_quality': 0,
                        'bot_critique': '',
                        'processing_status': f'error: {str(e)}',
                        'input_tokens': 0,
                        'output_tokens': 0,
                        'total_tokens': 0
                    }
                    update_progress(idx, error_result)

    except KeyboardInterrupt:
        print("\n\n⚠️ Processing interrupted by user!")
        print("Saving checkpoint before exit...")
        save_checkpoint(completed_indices, results_dict)
        pbar.close()
        print(f"Progress saved: {len(completed_indices)}/{total_rows} completed")
        print("You can resume later by running this cell again.")
        raise

    except Exception as e:
        print(f"\n\n❌ Error occurred: {e}")
        print("Saving checkpoint before exit...")
        save_checkpoint(completed_indices, results_dict)
        pbar.close()
        print("You can resume later by running this cell again.")
        raise

    # Final save
    pbar.close()
    save_checkpoint(completed_indices, results_dict)

    # Calculate timing
    elapsed_time = time.time() - start_time
    rows_processed_this_session = len(rows_to_process)
    rate = rows_processed_this_session / elapsed_time if elapsed_time > 0 else 0

    print("\n" + "="*60)
    print("✅ EXTRACTION COMPLETE!")
    print("="*60)
    print(f"Total processed: {len(completed_indices)} rows")
    print(f"This session: {rows_processed_this_session} rows in {elapsed_time:.1f}s")
    print(f"Processing rate: {rate:.2f} conversations/second")


# =============================================================================
# CELL 9B: BATCH MODE (Gemini Batch API)
# =============================================================================

elif PROCESSING_MODE == "batch":
    print(f"\n📦 BATCH MODE — Gemini Batch API (50% cost savings)")
    print("-"*60)

    results_dict = {}

    # ------------------------------------------------------------------
    # Check if a batch job was already submitted (resume support)
    # ------------------------------------------------------------------
    existing_job_name = None
    if os.path.exists(BATCH_JOB_FILE):
        try:
            with open(BATCH_JOB_FILE, 'r') as f:
                job_info = json.load(f)
            existing_job_name = job_info.get('job_name')
            print(f"📋 Found existing batch job: {existing_job_name}")
        except Exception as e:
            print(f"⚠️ Could not read batch job file: {e}")
            existing_job_name = None

    # ------------------------------------------------------------------
    # Step 1: Prepare JSONL & Submit (skip if job already exists)
    # ------------------------------------------------------------------
    if existing_job_name is None:
        print("\n📝 Step 1: Preparing batch requests JSONL file...")

        # Build the JSONL file
        request_count = 0
        with open(BATCH_JSONL_FILE, 'w', encoding='utf-8') as f:
            for idx in range(total_rows):
                row = df.iloc[idx]
                conversation_text = str(row.get('full_conversation', ''))
                user_content = EXTRACT_USER_PROMPT_TEMPLATE.format(
                    conversation=conversation_text
                )

                # Each line is a JSON object with key + GenerateContentRequest
                # NOTE: Field names use camelCase (raw REST API format) for Batch API compatibility
                batch_line = {
                    "key": f"row-{idx}",
                    "request": {
                        "contents": [
                            {
                                "parts": [{"text": user_content}],
                                "role": "user"
                            }
                        ],
                        "systemInstruction": {
                            "parts": [{"text": EXTRACT_SYSTEM_PROMPT}]
                        },
                        "generationConfig": {
                            "temperature": TEMPERATURE,
                            "maxOutputTokens": MAX_TOKENS,
                            "responseMimeType": "application/json"
                        }
                    }
                }
                f.write(json.dumps(batch_line, ensure_ascii=False) + "\n")
                request_count += 1

        file_size_mb = os.path.getsize(BATCH_JSONL_FILE) / (1024 * 1024)
        print(f"   ✅ Created {BATCH_JSONL_FILE}: {request_count} requests, {file_size_mb:.2f} MB")

        # Upload JSONL via File API
        print("\n📤 Step 2: Uploading JSONL file to Gemini File API...")
        uploaded_file = client.files.upload(
            file=BATCH_JSONL_FILE,
            config=genai_types.UploadFileConfig(
                display_name=f"extraction-batch-{MODEL_ID}",
                mime_type="jsonl"
            )
        )
        print(f"   ✅ Uploaded: {uploaded_file.name}")

        # Submit batch job
        print(f"\n🚀 Step 3: Submitting batch job with model '{MODEL_ID}'...")
        batch_job = client.batches.create(
            model=MODEL_ID,
            src=uploaded_file.name,
            config={
                'display_name': f"extraction-{MODEL_ID}-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
            },
        )
        job_name = batch_job.name
        print(f"   ✅ Batch job created: {job_name}")

        # Save job name for resume
        with open(BATCH_JOB_FILE, 'w') as f:
            json.dump({
                'job_name': job_name,
                'model_id': MODEL_ID,
                'total_rows': total_rows,
                'submitted_at': datetime.now().isoformat()
            }, f, indent=2)
        print(f"   💾 Job info saved to: {BATCH_JOB_FILE}")

    else:
        job_name = existing_job_name
        print(f"   ⏩ Skipping submission — using existing job: {job_name}")

    # ------------------------------------------------------------------
    # Step 4: Poll for completion
    # ------------------------------------------------------------------
    print(f"\n⏳ Step 4: Polling batch job status (every {BATCH_POLL_INTERVAL}s)...")
    print("   Press Ctrl+C to stop polling. You can resume later.")

    completed_states = {
        'JOB_STATE_SUCCEEDED',
        'JOB_STATE_FAILED',
        'JOB_STATE_CANCELLED',
        'JOB_STATE_EXPIRED',
    }

    # Use clear_output in notebooks to prevent output bloat
    try:
        from IPython.display import clear_output
        _is_notebook = True
    except ImportError:
        _is_notebook = False

    try:
        poll_start = time.time()
        poll_count = 0
        while True:
            batch_job = client.batches.get(name=job_name)
            state = batch_job.state.name if hasattr(batch_job.state, 'name') else str(batch_job.state)
            elapsed = time.time() - poll_start
            poll_count += 1

            # In notebooks: clear and rewrite to prevent output bloat
            if _is_notebook and poll_count % 5 == 0:
                clear_output(wait=True)
                print(f"⏳ Polling batch job: {job_name}")
                print(f"   Polls so far: {poll_count}")

            # Always show current status
            print(f"   [{datetime.now().strftime('%H:%M:%S')}] State: {state} | Elapsed: {elapsed/60:.1f} min")

            if state in completed_states:
                break

            time.sleep(BATCH_POLL_INTERVAL)

        # Final clear summary
        if _is_notebook:
            clear_output(wait=True)
        print(f"\n⏳ Polling complete after {poll_count} polls ({elapsed/60:.1f} min)")
        print(f"   Final state: {state}")

    except KeyboardInterrupt:
        print(f"\n\n⚠️ Polling interrupted! Job '{job_name}' is still running.")
        print(f"   Run this cell again to resume polling.")
        print(f"   Job info saved in: {BATCH_JOB_FILE}")
        raise

    # ------------------------------------------------------------------
    # Step 5: Download & Parse Results
    # ------------------------------------------------------------------
    final_state = batch_job.state.name if hasattr(batch_job.state, 'name') else str(batch_job.state)

    if final_state == 'JOB_STATE_SUCCEEDED':
        print(f"\n✅ Batch job SUCCEEDED!")
        print("📥 Step 5: Downloading and parsing results...")

        # Save results to disk first (safer for large datasets — avoids OOM)
        BATCH_RAW_RESULTS_FILE = BATCH_JSONL_FILE.replace('.jsonl', '_results.jsonl')

        # Determine result source: file or inline
        if hasattr(batch_job, 'dest') and batch_job.dest and hasattr(batch_job.dest, 'file_name') and batch_job.dest.file_name:
            # Results are in a file — download to disk
            result_file_name = batch_job.dest.file_name
            print(f"   Downloading result file: {result_file_name}")
            file_content = client.files.download(file=result_file_name)
            with open(BATCH_RAW_RESULTS_FILE, 'wb') as f:
                f.write(file_content)
            print(f"   💾 Saved raw results to: {BATCH_RAW_RESULTS_FILE}")

        elif hasattr(batch_job, 'dest') and batch_job.dest and hasattr(batch_job.dest, 'inlined_responses') and batch_job.dest.inlined_responses:
            # Results are inline — save to disk first
            print(f"   Processing inline responses...")
            with open(BATCH_RAW_RESULTS_FILE, 'w', encoding='utf-8') as f:
                for resp in batch_job.dest.inlined_responses:
                    if resp.response:
                        f.write(json.dumps({
                            "key": getattr(resp, 'key', ''),
                            "response": {"text": resp.response.text}
                        }, ensure_ascii=False) + '\n')
            print(f"   💾 Saved inline results to: {BATCH_RAW_RESULTS_FILE}")
        else:
            print("❌ No results found in batch job response!")
            BATCH_RAW_RESULTS_FILE = None

        # Read results from disk for parsing
        if BATCH_RAW_RESULTS_FILE and os.path.exists(BATCH_RAW_RESULTS_FILE):
            with open(BATCH_RAW_RESULTS_FILE, 'r', encoding='utf-8') as f:
                result_lines = [line.strip() for line in f if line.strip()]
        else:
            result_lines = []

        # Parse each result line and map back to row indices
        print(f"   Parsing {len(result_lines)} result lines...")
        parsed_count = 0
        error_count = 0

        for line in result_lines:
            try:
                result_obj = json.loads(line)
                key = result_obj.get("key", "")

                # Extract row index from key "row-{idx}"
                if key.startswith("row-"):
                    idx = int(key.split("-", 1)[1])
                else:
                    continue

                # Get the response text
                response_data = result_obj.get("response", {})
                if isinstance(response_data, dict):
                    # File-based results: response contains candidates
                    candidates = response_data.get("candidates", [])
                    if candidates:
                        parts = candidates[0].get("content", {}).get("parts", [])
                        response_text = parts[0].get("text", "") if parts else ""
                    else:
                        response_text = response_data.get("text", "")
                else:
                    response_text = str(response_data)

                # Parse & validate JSON from LLM response
                parsed_result = parse_json_response(response_text)
                flat_result = flatten_extraction_result(parsed_result)

                # Extract token counts from usage metadata if available
                usage = response_data.get("usageMetadata", {})
                input_tokens = usage.get("promptTokenCount", 0)
                output_tokens = usage.get("candidatesTokenCount", 0)

                conversation_id = df.iloc[idx].get('general_chat_id', 'unknown')

                results_dict[idx] = {
                    'conversation_id': conversation_id,
                    **flat_result,
                    'processing_status': flat_result.get('extraction_status', 'success'),
                    'input_tokens': input_tokens,
                    'output_tokens': output_tokens,
                    'total_tokens': input_tokens + output_tokens,
                }
                parsed_count += 1

            except Exception as e:
                error_count += 1
                # Try to extract idx from the key if possible
                try:
                    idx = int(key.split("-", 1)[1])
                    conversation_id = df.iloc[idx].get('general_chat_id', 'unknown')
                except:
                    idx = -1
                    conversation_id = 'unknown'

                if idx >= 0:
                    results_dict[idx] = {
                        'conversation_id': conversation_id,
                        'conversation_summary': 'EXTRACTION_ERROR',
                        'suspected_religion': 'ERROR',
                        'is_existing_muslim': False,
                        'user_language': 'ERROR',
                        'conversation_type': 'ERROR',
                        'engagement_score': 0,
                        'engagement_flow': 'ERROR',
                        'topics_discussed': '[]',
                        'key_blocker': 'ERROR',
                        'user_objections': '[]',
                        'start_mood': 'ERROR',
                        'end_mood': 'ERROR',
                        'user_intent': 'ERROR',
                        'conversion_funnel': 'ERROR',
                        'script_dumping': False,
                        'response_quality': 0,
                        'bot_critique': 'ERROR',
                        'processing_status': f'batch_parse_error: {str(e)}',
                        'input_tokens': 0,
                        'output_tokens': 0,
                        'total_tokens': 0,
                    }

        # Fill in any missing rows (requests that got no response)
        for idx in range(total_rows):
            if idx not in results_dict:
                results_dict[idx] = {
                    'conversation_id': df.iloc[idx].get('general_chat_id', 'unknown'),
                    'conversation_summary': 'EXTRACTION_ERROR',
                    'suspected_religion': 'ERROR',
                    'is_existing_muslim': False,
                    'user_language': 'ERROR',
                    'conversation_type': 'ERROR',
                    'engagement_score': 0,
                    'engagement_flow': 'ERROR',
                    'topics_discussed': '[]',
                    'key_blocker': 'ERROR',
                    'user_objections': '[]',
                    'start_mood': 'ERROR',
                    'end_mood': 'ERROR',
                    'user_intent': 'ERROR',
                    'conversion_funnel': 'ERROR',
                    'script_dumping': False,
                    'response_quality': 0,
                    'bot_critique': 'ERROR',
                    'processing_status': 'batch_no_response',
                    'input_tokens': 0,
                    'output_tokens': 0,
                    'total_tokens': 0,
                }

        elapsed_time = time.time() - start_time

        print(f"\n   ✅ Parsed: {parsed_count} | Errors: {error_count} | Missing: {total_rows - parsed_count - error_count}")
        print(f"   Total time (including wait): {elapsed_time/60:.1f} minutes")

        # Clean up batch job file
        if os.path.exists(BATCH_JOB_FILE):
            os.remove(BATCH_JOB_FILE)
            print(f"   🗑️ Cleaned up {BATCH_JOB_FILE}")

    else:
        # Job failed / cancelled / expired
        print(f"\n❌ Batch job ended with state: {final_state}")
        if hasattr(batch_job, 'error') and batch_job.error:
            print(f"   Error: {batch_job.error}")
        print(f"   You may need to resubmit. Delete '{BATCH_JOB_FILE}' to start fresh.")

        # Fill results_dict with error entries so downstream cells don't crash
        for idx in range(total_rows):
            results_dict[idx] = {
                'conversation_id': df.iloc[idx].get('general_chat_id', 'unknown'),
                'conversation_summary': 'EXTRACTION_ERROR',
                'suspected_religion': 'ERROR',
                'is_existing_muslim': False,
                'user_language': 'ERROR',
                'conversation_type': 'ERROR',
                'engagement_score': 0,
                'engagement_flow': 'ERROR',
                'topics_discussed': '[]',
                'key_blocker': 'ERROR',
                'user_objections': '[]',
                'start_mood': 'ERROR',
                'end_mood': 'ERROR',
                'user_intent': 'ERROR',
                'conversion_funnel': 'ERROR',
                'script_dumping': False,
                'response_quality': 0,
                'bot_critique': 'ERROR',
                'processing_status': f'batch_{final_state.lower()}',
                'input_tokens': 0,
                'output_tokens': 0,
                'total_tokens': 0,
            }

        elapsed_time = time.time() - start_time

    print("\n" + "="*60)
    print("✅ BATCH PROCESSING COMPLETE!")
    print("="*60)




⏳ Polling complete after 1 polls (0.0 min)
   Final state: JOB_STATE_SUCCEEDED

✅ Batch job SUCCEEDED!
📥 Step 5: Downloading and parsing results...
   💾 Saved raw results to: batch_requests_results.jsonl
   Parsing 12448 result lines...

   ✅ Parsed: 12448 | Errors: 0 | Missing: 0
   Total time (including wait): 0.3 minutes
   🗑️ Cleaned up batch_job_gemini_3_1_flash_lite_preview.json

✅ BATCH PROCESSING COMPLETE!


In [10]:

# =============================================================================
# CELL 10: Merge Results and Save to CSV
# =============================================================================

print("="*60)
print("MERGING RESULTS")
print("="*60)

# Convert results_dict to ordered list matching DataFrame
results = [results_dict[idx] for idx in range(total_rows)]

# Merge with original dataframe
print("\nMerging results with original data...")

# Define new columns to add
new_columns = [
    'conversation_summary',
    'suspected_religion',
    'is_existing_muslim',
    'user_language',
    'conversation_type',
    'engagement_score',
    'engagement_flow',
    'topics_discussed',
    'key_blocker',
    'user_objections',
    'start_mood',
    'end_mood',
    'user_intent',
    'conversion_funnel',
    'script_dumping',
    'response_quality',
    'bot_critique',
    'processing_status',
    'input_tokens',
    'output_tokens',
    'total_tokens'
]

# Add new columns to original dataframe
for col in new_columns:
    df[col] = [results_dict.get(idx, {}).get(col, '') for idx in range(total_rows)]

# Convert engagement_score to numeric
df['engagement_score'] = pd.to_numeric(df['engagement_score'], errors='coerce').fillna(0).astype(int)

# Convert response_quality to numeric
df['response_quality'] = pd.to_numeric(df['response_quality'], errors='coerce').fillna(0).astype(int)

# Convert script_dumping to boolean
df['script_dumping'] = df['script_dumping'].apply(lambda x: x if isinstance(x, bool) else str(x).lower() == 'true')

# Convert is_existing_muslim to boolean
df['is_existing_muslim'] = df['is_existing_muslim'].apply(lambda x: x if isinstance(x, bool) else str(x).lower() == 'true')

# Save final output
print(f"\nSaving to: {OUTPUT_FILE}")
df.to_csv(OUTPUT_FILE, index=False)
print("✅ CSV file saved successfully!")

# Clean up checkpoint
delete_checkpoint()



MERGING RESULTS

Merging results with original data...

Saving to: full_conversation_points_extraction.csv
✅ CSV file saved successfully!


In [11]:

# =============================================================================
# CELL 11: Save Results to JSON File
# =============================================================================

print("\n" + "="*60)
print("SAVING RESULTS TO JSON")
print("="*60)

# Define JSON output filename
JSON_OUTPUT_FILE = f"full_conversation_points_extraction.json"

# Prepare results for JSON export
json_results = []

for idx, row in df.iterrows():
    # Build structured JSON for each conversation
    conv_result = {
        "conversation_id": row.get('general_chat_id', ''),
        "conversation_summary": row.get('conversation_summary', ''),
        
        "user_demographics": {
            "suspected_religion": row.get('suspected_religion', ''),
            "is_existing_muslim": bool(row.get('is_existing_muslim', False)),
            "language": row.get('user_language', '')
        },
        
        "conversation_type": row.get('conversation_type', ''),
        
        "engagement_quality": {
            "score": int(row.get('engagement_score', 0)),
            "flow": row.get('engagement_flow', '')
        },
        
        "theological_profile": {
            "topics_discussed": json.loads(row.get('topics_discussed', '[]')) if isinstance(row.get('topics_discussed'), str) else row.get('topics_discussed', []),
            "key_blocker": row.get('key_blocker', ''),
            "user_objections": json.loads(row.get('user_objections', '[]')) if isinstance(row.get('user_objections'), str) else row.get('user_objections', [])
        },
        
        "emotional_trajectory": {
            "start_mood": row.get('start_mood', ''),
            "end_mood": row.get('end_mood', '')
        },
        
        "intent_and_funnel": {
            "user_intent": row.get('user_intent', ''),
            "conversion_funnel": row.get('conversion_funnel', '')
        },
        
        "bot_quality_audit": {
            "script_dumping": bool(row.get('script_dumping', False)),
            "response_quality": int(row.get('response_quality', 0)),
            "critique": row.get('bot_critique', '')
        },
        
        "processing_metadata": {
            "status": row.get('processing_status', ''),
            "input_tokens": int(row.get('input_tokens', 0)),
            "output_tokens": int(row.get('output_tokens', 0)),
            "total_tokens": int(row.get('total_tokens', 0))
        }
    }
    
    json_results.append(conv_result)

# Create final JSON structure
json_output = {
    "metadata": {
        "provider": PROVIDER,
        "model_id": MODEL_ID,
        "total_conversations": len(json_results),
        "successful": len(df[df['processing_status'] == 'success']),
        "failed": len(df[df['processing_status'] != 'success']),
        "generated_at": datetime.now().isoformat()
    },
    "results": json_results
}

# Save to JSON file
with open(JSON_OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(json_output, f, indent=2, ensure_ascii=False)

print(f"✅ JSON file saved to: {JSON_OUTPUT_FILE}")
print(f"   Total conversations: {len(json_results)}")
print(f"   File size: {os.path.getsize(JSON_OUTPUT_FILE) / 1024:.2f} KB")




SAVING RESULTS TO JSON
✅ JSON file saved to: full_conversation_points_extraction.json
   Total conversations: 12448
   File size: 20925.69 KB


In [12]:

# =============================================================================
# CELL 12: Token Statistics
# =============================================================================

print("\n" + "="*60)
print(f"TOKEN STATISTICS FOR {PROVIDER.upper()} / {MODEL_ID.upper()}")
print("="*60)

# Convert token columns to numeric
token_columns = ['input_tokens', 'output_tokens', 'total_tokens']

for col in token_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Filter successful rows only
successful_df = df[df['processing_status'] == 'success']

print(f"\nSuccessful conversations: {len(successful_df)} / {len(df)}")

# Calculate statistics
def calc_stats(series):
    """Calculate mean and median for a series."""
    values = series[series > 0].tolist()
    if not values:
        return {'mean': 0, 'median': 0, 'min': 0, 'max': 0, 'total': 0}
    return {
        'mean': round(statistics.mean(values), 2),
        'median': round(statistics.median(values), 2),
        'min': min(values),
        'max': max(values),
        'total': sum(values)
    }

# Token statistics
print("\n📊 TOKEN USAGE:")
input_stats = calc_stats(successful_df['input_tokens'])
output_stats = calc_stats(successful_df['output_tokens'])
total_stats = calc_stats(successful_df['total_tokens'])

print(f"   Input tokens  - Mean: {input_stats['mean']:,.0f} | Median: {input_stats['median']:,.0f} | Total: {input_stats['total']:,}")
print(f"   Output tokens - Mean: {output_stats['mean']:,.0f} | Median: {output_stats['median']:,.0f} | Total: {output_stats['total']:,}")
print(f"   Total tokens  - Mean: {total_stats['mean']:,.0f} | Median: {total_stats['median']:,.0f} | Total: {total_stats['total']:,}")

# Grand totals
grand_total_input = input_stats['total']
grand_total_output = output_stats['total']
print(f"\n🔢 GRAND TOTALS:")
print(f"   Total input tokens:  {grand_total_input:,}")
print(f"   Total output tokens: {grand_total_output:,}")
print(f"   Total all tokens:    {grand_total_input + grand_total_output:,}")

# Save statistics to JSON
stats_report = {
    "provider": PROVIDER,
    "model_id": MODEL_ID,
    "num_workers": NUM_WORKERS,
    "total_conversations": len(df),
    "successful_conversations": len(successful_df),
    "tokens": {
        "input": input_stats,
        "output": output_stats,
        "total": total_stats
    },
    "grand_totals": {
        "input_tokens": grand_total_input,
        "output_tokens": grand_total_output,
        "all_tokens": grand_total_input + grand_total_output
    },
    "timestamp": datetime.now().isoformat()
}

stats_file = f"token_stats_{MODEL_ID.replace('.', '_').replace('-', '_')}.json"
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats_report, f, indent=2)
print(f"\n✅ Token statistics saved to: {stats_file}")




TOKEN STATISTICS FOR GEMINI / GEMINI-3.1-FLASH-LITE-PREVIEW

Successful conversations: 12447 / 12448

📊 TOKEN USAGE:
   Input tokens  - Mean: 4,235 | Median: 3,065 | Total: 52,713,640
   Output tokens - Mean: 375 | Median: 365 | Total: 4,671,864
   Total tokens  - Mean: 4,610 | Median: 3,440 | Total: 57,385,504

🔢 GRAND TOTALS:
   Total input tokens:  52,713,640
   Total output tokens: 4,671,864
   Total all tokens:    57,385,504

✅ Token statistics saved to: token_stats_gemini_3_1_flash_lite_preview.json


In [13]:

# =============================================================================
# CELL 13: Cost Estimation (Dynamic - Provider & Model Aware)
# =============================================================================

print("\n" + "="*60)
print(f"COST ESTIMATION ({PROVIDER.upper()} / {MODEL_ID})")
print("="*60)

# Lookup pricing dynamically
if MODEL_ID in MODEL_PRICING:
    pricing = MODEL_PRICING[MODEL_ID]
    PRICE_INPUT = pricing["input"]
    PRICE_OUTPUT = pricing["output"]
else:
    print(f"⚠️  Model '{MODEL_ID}' not in pricing dictionary. Showing $0 costs.")
    PRICE_INPUT = 0
    PRICE_OUTPUT = 0

# Apply 50% batch discount
if PROCESSING_MODE == "batch":
    PRICE_INPUT *= 0.5
    PRICE_OUTPUT *= 0.5
    print(f"\n🏷️ BATCH MODE DISCOUNT: 50% off applied!")

# Calculate costs
input_cost = (grand_total_input / 1_000_000) * PRICE_INPUT
output_cost = (grand_total_output / 1_000_000) * PRICE_OUTPUT
total_cost = input_cost + output_cost

# Per conversation cost (using mean)
per_conv_input = (input_stats['mean'] / 1_000_000) * PRICE_INPUT
per_conv_output = (output_stats['mean'] / 1_000_000) * PRICE_OUTPUT
per_conv_total = per_conv_input + per_conv_output

print(f"\n💰 Pricing for {MODEL_ID}:")
print(f"   Input:  ${PRICE_INPUT:.2f} / 1M tokens")
print(f"   Output: ${PRICE_OUTPUT:.2f} / 1M tokens")

print(f"\n💵 TOTAL COST (this run):")
print(f"   Input cost:  ${input_cost:.4f}")
print(f"   Output cost: ${output_cost:.4f}")
print(f"   Total cost:  ${total_cost:.4f}")

print(f"\n💵 ESTIMATED COST PER CONVERSATION (using mean):")
print(f"   Per conversation: ${per_conv_total:.6f}")
print(f"   Per 1,000 conversations: ${per_conv_total * 1000:.4f}")
print(f"   Per 10,000 conversations: ${per_conv_total * 10000:.2f}")

# Add cost to stats report
stats_report["cost_estimation"] = {
    "pricing_per_1m_tokens": {"input": PRICE_INPUT, "output": PRICE_OUTPUT},
    "total_cost": {
        "input": round(input_cost, 6),
        "output": round(output_cost, 6),
        "total": round(total_cost, 6)
    },
    "per_conversation_cost": {
        "mean": round(per_conv_total, 8),
        "per_1000": round(per_conv_total * 1000, 4),
        "per_10000": round(per_conv_total * 10000, 2)
    }
}

# Update stats file with cost info
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats_report, f, indent=2)




COST ESTIMATION (GEMINI / gemini-3.1-flash-lite-preview)

🏷️ BATCH MODE DISCOUNT: 50% off applied!

💰 Pricing for gemini-3.1-flash-lite-preview:
   Input:  $0.12 / 1M tokens
   Output: $0.75 / 1M tokens

💵 TOTAL COST (this run):
   Input cost:  $6.5892
   Output cost: $3.5039
   Total cost:  $10.0931

💵 ESTIMATED COST PER CONVERSATION (using mean):
   Per conversation: $0.000811
   Per 1,000 conversations: $0.8109
   Per 10,000 conversations: $8.11


In [14]:

# =============================================================================
# CELL 14: Final Summary & Distributions
# =============================================================================

print("\n" + "="*60)
print("FINAL SUMMARY & DISTRIBUTIONS")
print("="*60)

print(f"\n📊 Provider: {PROVIDER.upper()}")
print(f"📊 Model: {MODEL_ID}")
print(f"📊 Workers: {NUM_WORKERS}")
print(f"📊 Total conversations processed: {len(df)}")

# Processing status breakdown
print("\n📈 Processing Status:")
status_counts = df['processing_status'].value_counts()
for status, count in status_counts.items():
    pct = (count / len(df)) * 100
    print(f"   {status}: {count} ({pct:.1f}%)")

# Conversation Type distribution
print("\n📂 Conversation Type Distribution:")
type_counts = df['conversation_type'].value_counts()
for ctype, count in type_counts.items():
    pct = (count / len(df)) * 100
    print(f"   {ctype}: {count} ({pct:.1f}%)")

# User Intent distribution
print("\n🎯 User Intent Distribution:")
intent_counts = df['user_intent'].value_counts()
for intent, count in intent_counts.items():
    pct = (count / len(df)) * 100
    print(f"   {intent}: {count} ({pct:.1f}%)")

# Existing Muslim breakdown
print("\n🕌 Existing Muslim Breakdown:")
muslim_counts = df['is_existing_muslim'].value_counts()
for val, count in muslim_counts.items():
    pct = (count / len(df)) * 100
    label = "Muslim" if val else "Non-Muslim / Unknown"
    print(f"   {label}: {count} ({pct:.1f}%)")

# Conversion Funnel distribution
print("\n📊 Conversion Funnel Distribution:")
funnel_counts = df['conversion_funnel'].value_counts()
for funnel, count in funnel_counts.items():
    pct = (count / len(df)) * 100
    print(f"   {funnel}: {count} ({pct:.1f}%)")

# Engagement Score distribution
print("\n⭐ Engagement Score Distribution:")
score_counts = df['engagement_score'].value_counts().sort_index()
for score, count in score_counts.items():
    pct = (count / len(df)) * 100
    bar = "█" * int(pct / 5)  # Visual bar
    print(f"   Score {score}: {count} ({pct:.1f}%) {bar}")

# Response Quality distribution
print("\n🤖 Bot Response Quality Distribution:")
rq_counts = df['response_quality'].value_counts().sort_index()
for rq, count in rq_counts.items():
    pct = (count / len(df)) * 100
    bar = "█" * int(pct / 5)
    print(f"   Quality {rq}: {count} ({pct:.1f}%) {bar}")

# Suspected Religion distribution
print("\n🙏 Suspected Religion Distribution:")
religion_counts = df['suspected_religion'].value_counts()
for religion, count in religion_counts.head(10).items():
    pct = (count / len(df)) * 100
    print(f"   {religion}: {count} ({pct:.1f}%)")

# Engagement Flow distribution
print("\n🔄 Engagement Flow Distribution:")
flow_counts = df['engagement_flow'].value_counts()
for flow, count in flow_counts.items():
    pct = (count / len(df)) * 100
    print(f"   {flow}: {count} ({pct:.1f}%)")

# Script Dumping analysis
print("\n🤖 Bot Script Dumping:")
dumping_counts = df['script_dumping'].value_counts()
for dumping, count in dumping_counts.items():
    pct = (count / len(df)) * 100
    label = "Yes (Script Dumping)" if dumping else "No (Responsive)"
    print(f"   {label}: {count} ({pct:.1f}%)")

# Key Blockers (top 10)
print("\n🚧 Top Key Blockers:")
blocker_counts = df['key_blocker'].value_counts()
for blocker, count in blocker_counts.head(10).items():
    pct = (count / len(df)) * 100
    print(f"   {blocker}: {count} ({pct:.1f}%)")

# Mood Transitions
print("\n😊 Start Mood Distribution (Top 5):")
start_mood_counts = df['start_mood'].value_counts()
for mood, count in start_mood_counts.head(5).items():
    pct = (count / len(df)) * 100
    print(f"   {mood}: {count} ({pct:.1f}%)")

print("\n😔 End Mood Distribution (Top 5):")
end_mood_counts = df['end_mood'].value_counts()
for mood, count in end_mood_counts.head(5).items():
    pct = (count / len(df)) * 100
    print(f"   {mood}: {count} ({pct:.1f}%)")

# Preview of results
print("\n" + "-"*60)
print("📋 Preview of results:")
preview_cols = [
    "general_chat_id", 
    "user_intent", 
    "conversation_type",
    "conversion_funnel", 
    "engagement_score",
    "suspected_religion",
    "is_existing_muslim"
]
available_cols = [c for c in preview_cols if c in df.columns]
print(df[available_cols].head(10).to_string())




FINAL SUMMARY & DISTRIBUTIONS

📊 Provider: GEMINI
📊 Model: gemini-3.1-flash-lite-preview
📊 Workers: 16
📊 Total conversations processed: 12448

📈 Processing Status:
   success: 12447 (100.0%)
   failed: 1 (0.0%)

📂 Conversation Type Distribution:
   Dawah: 3419 (27.5%)
   Minimal/No Engagement: 2834 (22.8%)
   Islamic Guidance: 2814 (22.6%)
   Theological Debate: 1354 (10.9%)
   Dawah Training: 912 (7.3%)
   Off-Topic: 783 (6.3%)
   Content Assistance: 269 (2.2%)
   Emotional Support: 54 (0.4%)
   Muslim Learner: 8 (0.1%)
   ERROR: 1 (0.0%)

🎯 User Intent Distribution:
   Passive Listener: 2798 (22.5%)
   Muslim Learner: 2438 (19.6%)
   Genuine Seeker: 2013 (16.2%)
   Greeting Only: 1925 (15.5%)
   Challenger: 1546 (12.4%)
   Off-Topic User: 911 (7.3%)
   Troll/Spam: 340 (2.7%)
   Conversion Interest: 262 (2.1%)
   Dawah Training: 192 (1.5%)
   Content Assistance: 14 (0.1%)
   Minimal/No Engagement: 8 (0.1%)
   ERROR: 1 (0.0%)

🕌 Existing Muslim Breakdown:
   Non-Muslim / Unknown: 7847

In [15]:

# =============================================================================
# CELL 15: Performance Summary & Output Files
# =============================================================================

print("\n" + "="*60)
print("🏁 FINAL PERFORMANCE SUMMARY")
print("="*60)

print(f"\n⚡ PARALLEL PROCESSING STATS:")
print(f"   Provider: {PROVIDER.upper()}")
print(f"   Model: {MODEL_ID}")
print(f"   Workers used: {NUM_WORKERS}")
print(f"   Total time: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
print(f"   Processing rate: {rate:.2f} conversations/second")

print(f"\n📁 OUTPUT FILES:")
print(f"   ├── {OUTPUT_FILE} (CSV)")
print(f"   ├── {JSON_OUTPUT_FILE} (JSON)")
print(f"   └── {stats_file} (Token Stats)")

print(f"\n📊 DATA SUMMARY:")
print(f"   Total conversations: {len(df)}")
success_count = len(df[df['processing_status'] == 'success'])
fail_count = len(df) - success_count
print(f"   Successful: {success_count} ({success_count/len(df)*100:.1f}%)")
print(f"   Failed: {fail_count} ({fail_count/len(df)*100:.1f}%)")

print(f"\n💰 COST SUMMARY:")
print(f"   Total cost: ${total_cost:.4f}")
print(f"   Cost per conversation: ${per_conv_total:.6f}")

print("\n" + "="*60)
print("✅ ALL DONE!")
print("="*60)


🏁 FINAL PERFORMANCE SUMMARY

⚡ PARALLEL PROCESSING STATS:
   Provider: GEMINI
   Model: gemini-3.1-flash-lite-preview
   Workers used: 16
   Total time: 15.5 seconds (0.3 minutes)


NameError: name 'rate' is not defined